# Oil Spill Segmentation Model (Step 2)

This notebook implements the second step of our two-stage framework:
**Pixel-level segmentation to delineate oil spill boundaries in SAR imagery**

## Architecture:
- U-Net with ResNet/EfficientNet encoder
- Feature Pyramid Network (FPN) decoder
- Advanced loss functions (Dice, Focal, Combined)
- Multi-scale training and inference

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR

import segmentation_models_pytorch as smp
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import jaccard_score, f1_score
import pandas as pd
from tqdm import tqdm
import json
import os
from pathlib import Path
import cv2
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Segmentation Model Architecture

In [ ]:
class OilSpillSegmentationModel(nn.Module):
    """Oil spill segmentation model using U-Net architecture"""
    
    def __init__(self, architecture='unet', encoder_name='resnet50', 
                 encoder_weights='imagenet', in_channels=3, classes=1, activation=None):
        super(OilSpillSegmentationModel, self).__init__()
        
        self.architecture = architecture
        self.encoder_name = encoder_name
        
        if architecture == 'unet':
            self.model = smp.Unet(
                encoder_name=encoder_name,
                encoder_weights=encoder_weights,
                in_channels=in_channels,
                classes=classes,
                activation=activation
            )
        elif architecture == 'unetplusplus':
            self.model = smp.UnetPlusPlus(
                encoder_name=encoder_name,
                encoder_weights=encoder_weights,
                in_channels=in_channels,
                classes=classes,
                activation=activation
            )
        elif architecture == 'fpn':
            self.model = smp.FPN(
                encoder_name=encoder_name,
                encoder_weights=encoder_weights,
                in_channels=in_channels,
                classes=classes,
                activation=activation
            )
        elif architecture == 'pspnet':
            self.model = smp.PSPNet(
                encoder_name=encoder_name,
                encoder_weights=encoder_weights,
                in_channels=in_channels,
                classes=classes,
                activation=activation
            )
        elif architecture == 'deeplabv3plus':
            self.model = smp.DeepLabV3Plus(
                encoder_name=encoder_name,
                encoder_weights=encoder_weights,
                in_channels=in_channels,
                classes=classes,
                activation=activation
            )
        else:
            raise ValueError(f"Unsupported architecture: {architecture}")
    
    def forward(self, x):
        return self.model(x)

class AttentionUNet(nn.Module):
    """Custom U-Net with attention mechanism for oil spill segmentation"""
    
    def __init__(self, in_channels=3, out_channels=1, features=64):
        super(AttentionUNet, self).__init__()
        
        # Encoder (Contracting path)
        self.encoder1 = self._conv_block(in_channels, features)
        self.pool1 = nn.MaxPool2d(2, 2)
        
        self.encoder2 = self._conv_block(features, features * 2)
        self.pool2 = nn.MaxPool2d(2, 2)
        
        self.encoder3 = self._conv_block(features * 2, features * 4)
        self.pool3 = nn.MaxPool2d(2, 2)
        
        self.encoder4 = self._conv_block(features * 4, features * 8)
        self.pool4 = nn.MaxPool2d(2, 2)
        
        # Bottleneck
        self.bottleneck = self._conv_block(features * 8, features * 16)
        
        # Decoder (Expansive path)
        self.upconv4 = nn.ConvTranspose2d(features * 16, features * 8, 2, 2)
        self.attention4 = AttentionBlock(features * 8, features * 8, features * 4)
        self.decoder4 = self._conv_block(features * 16, features * 8)
        
        self.upconv3 = nn.ConvTranspose2d(features * 8, features * 4, 2, 2)
        self.attention3 = AttentionBlock(features * 4, features * 4, features * 2)
        self.decoder3 = self._conv_block(features * 8, features * 4)
        
        self.upconv2 = nn.ConvTranspose2d(features * 4, features * 2, 2, 2)
        self.attention2 = AttentionBlock(features * 2, features * 2, features)
        self.decoder2 = self._conv_block(features * 4, features * 2)
        
        self.upconv1 = nn.ConvTranspose2d(features * 2, features, 2, 2)
        self.attention1 = AttentionBlock(features, features, features // 2)
        self.decoder1 = self._conv_block(features * 2, features)
        
        # Output layer
        self.final = nn.Conv2d(features, out_channels, 1)
    
    def _conv_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        # Encoder
        enc1 = self.encoder1(x)
        enc2 = self.encoder2(self.pool1(enc1))
        enc3 = self.encoder3(self.pool2(enc2))
        enc4 = self.encoder4(self.pool3(enc3))
        
        # Bottleneck
        bottleneck = self.bottleneck(self.pool4(enc4))
        
        # Decoder with attention
        dec4 = self.upconv4(bottleneck)
        att4 = self.attention4(g=dec4, x=enc4)
        dec4 = torch.cat([att4, dec4], dim=1)
        dec4 = self.decoder4(dec4)
        
        dec3 = self.upconv3(dec4)
        att3 = self.attention3(g=dec3, x=enc3)
        dec3 = torch.cat([att3, dec3], dim=1)
        dec3 = self.decoder3(dec3)
        
        dec2 = self.upconv2(dec3)
        att2 = self.attention2(g=dec2, x=enc2)
        dec2 = torch.cat([att2, dec2], dim=1)
        dec2 = self.decoder2(dec2)
        
        dec1 = self.upconv1(dec2)
        att1 = self.attention1(g=dec1, x=enc1)
        dec1 = torch.cat([att1, dec1], dim=1)
        dec1 = self.decoder1(dec1)
        
        return self.final(dec1)

class AttentionBlock(nn.Module):
    """Attention block for U-Net"""
    
    def __init__(self, F_g, F_l, F_int):
        super(AttentionBlock, self).__init__()
        
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, 1, bias=True),
            nn.BatchNorm2d(F_int)
        )
        
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, 1, bias=True),
            nn.BatchNorm2d(F_int)
        )
        
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, 1, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi

def create_segmentation_model(config):
    """Create segmentation model based on configuration"""
    model_type = config.get('model_type', 'smp_unet')
    
    if model_type == 'smp_unet':
        model = OilSpillSegmentationModel(
            architecture=config.get('architecture', 'unet'),
            encoder_name=config.get('encoder_name', 'resnet50'),
            encoder_weights=config.get('encoder_weights', 'imagenet'),
            in_channels=config.get('in_channels', 3),
            classes=config.get('classes', 1),
            activation=config.get('activation', None)
        )
    elif model_type == 'attention_unet':
        model = AttentionUNet(
            in_channels=config.get('in_channels', 3),
            out_channels=config.get('classes', 1),
            features=config.get('features', 64)
        )
    else:
        raise ValueError(f"Unsupported model type: {model_type}")
    
    return model

## 2. Advanced Loss Functions for Segmentation

In [ ]:
class DiceLoss(nn.Module):
    """Dice Loss for segmentation"""
    
    def __init__(self, smooth=1e-6):
        super(DiceLoss, self).__init__()
        self.smooth = smooth
    
    def forward(self, predictions, targets):
        # Flatten tensors
        predictions = predictions.view(-1)
        targets = targets.view(-1)
        
        intersection = (predictions * targets).sum()
        dice = (2. * intersection + self.smooth) / (predictions.sum() + targets.sum() + self.smooth)
        
        return 1 - dice

class IoULoss(nn.Module):
    """IoU (Jaccard) Loss for segmentation"""
    
    def __init__(self, smooth=1e-6):
        super(IoULoss, self).__init__()
        self.smooth = smooth
    
    def forward(self, predictions, targets):
        # Flatten tensors
        predictions = predictions.view(-1)
        targets = targets.view(-1)
        
        intersection = (predictions * targets).sum()
        union = predictions.sum() + targets.sum() - intersection
        iou = (intersection + self.smooth) / (union + self.smooth)
        
        return 1 - iou

class FocalLoss(nn.Module):
    """Focal Loss for handling class imbalance in segmentation"""
    
    def __init__(self, alpha=1, gamma=2):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
    
    def forward(self, predictions, targets):
        bce_loss = F.binary_cross_entropy_with_logits(predictions, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
        return focal_loss.mean()

class TverskyLoss(nn.Module):
    """Tversky Loss - generalization of Dice Loss"""
    
    def __init__(self, alpha=0.3, beta=0.7, smooth=1e-6):
        super(TverskyLoss, self).__init__()
        self.alpha = alpha  # False positive penalty
        self.beta = beta    # False negative penalty
        self.smooth = smooth
    
    def forward(self, predictions, targets):
        # Flatten tensors
        predictions = predictions.view(-1)
        targets = targets.view(-1)
        
        TP = (predictions * targets).sum()
        FP = ((1 - targets) * predictions).sum()
        FN = (targets * (1 - predictions)).sum()
        
        tversky = (TP + self.smooth) / (TP + self.alpha * FP + self.beta * FN + self.smooth)
        
        return 1 - tversky

class CombinedLoss(nn.Module):
    """Combination of multiple loss functions"""
    
    def __init__(self, dice_weight=0.5, focal_weight=0.3, iou_weight=0.2):
        super(CombinedLoss, self).__init__()
        self.dice_loss = DiceLoss()
        self.focal_loss = FocalLoss()
        self.iou_loss = IoULoss()
        
        self.dice_weight = dice_weight
        self.focal_weight = focal_weight
        self.iou_weight = iou_weight
    
    def forward(self, predictions, targets):
        # Apply sigmoid to predictions for losses that expect probabilities
        predictions_prob = torch.sigmoid(predictions)
        
        dice = self.dice_loss(predictions_prob, targets)
        focal = self.focal_loss(predictions, targets)  # Uses logits
        iou = self.iou_loss(predictions_prob, targets)
        
        combined = (self.dice_weight * dice + 
                   self.focal_weight * focal + 
                   self.iou_weight * iou)
        
        return combined

def get_segmentation_loss(config):
    """Get loss function based on configuration"""
    loss_type = config.get('loss_type', 'combined')
    
    if loss_type == 'dice':
        return DiceLoss(smooth=config.get('smooth', 1e-6))
    elif loss_type == 'iou':
        return IoULoss(smooth=config.get('smooth', 1e-6))
    elif loss_type == 'focal':
        return FocalLoss(
            alpha=config.get('focal_alpha', 1),
            gamma=config.get('focal_gamma', 2)
        )
    elif loss_type == 'tversky':
        return TverskyLoss(
            alpha=config.get('tversky_alpha', 0.3),
            beta=config.get('tversky_beta', 0.7),
            smooth=config.get('smooth', 1e-6)
        )
    elif loss_type == 'combined':
        return CombinedLoss(
            dice_weight=config.get('dice_weight', 0.5),
            focal_weight=config.get('focal_weight', 0.3),
            iou_weight=config.get('iou_weight', 0.2)
        )
    elif loss_type == 'bce':
        return nn.BCEWithLogitsLoss()
    else:
        raise ValueError(f"Unsupported loss type: {loss_type}")

## 3. Segmentation Metrics

In [ ]:
def calculate_metrics(predictions, targets, threshold=0.5):
    """Calculate comprehensive segmentation metrics"""
    # Apply threshold to predictions
    predictions_binary = (predictions > threshold).float()
    
    # Flatten for metric calculation
    pred_flat = predictions_binary.view(-1).cpu().numpy()
    target_flat = targets.view(-1).cpu().numpy()
    
    # Calculate metrics
    intersection = (pred_flat * target_flat).sum()
    union = pred_flat.sum() + target_flat.sum() - intersection
    
    # IoU (Jaccard Index)
    iou = intersection / (union + 1e-8)
    
    # Dice Score
    dice = (2 * intersection) / (pred_flat.sum() + target_flat.sum() + 1e-8)
    
    # Pixel Accuracy
    pixel_acc = (pred_flat == target_flat).mean()
    
    # Precision, Recall, F1
    if pred_flat.sum() > 0:
        precision = intersection / pred_flat.sum()
    else:
        precision = 0.0
    
    if target_flat.sum() > 0:
        recall = intersection / target_flat.sum()
    else:
        recall = 0.0
    
    if precision + recall > 0:
        f1 = 2 * (precision * recall) / (precision + recall)
    else:
        f1 = 0.0
    
    return {
        'iou': iou,
        'dice': dice,
        'pixel_accuracy': pixel_acc,
        'precision': precision,
        'recall': recall,
        'f1_score': f1
    }

class SegmentationMetrics:
    """Segmentation metrics tracker"""
    
    def __init__(self):
        self.reset()
    
    def reset(self):
        self.metrics = {
            'iou': [],
            'dice': [],
            'pixel_accuracy': [],
            'precision': [],
            'recall': [],
            'f1_score': []
        }
    
    def update(self, predictions, targets, threshold=0.5):
        """Update metrics with batch results"""
        batch_metrics = calculate_metrics(predictions, targets, threshold)
        
        for key, value in batch_metrics.items():
            self.metrics[key].append(value)
    
    def get_average_metrics(self):
        """Get average metrics across all batches"""
        avg_metrics = {}
        for key, values in self.metrics.items():
            avg_metrics[key] = np.mean(values) if values else 0.0
        return avg_metrics

def visualize_segmentation_results(images, targets, predictions, num_samples=4):
    """Visualize segmentation results"""
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4 * num_samples))
    
    for i in range(min(num_samples, len(images))):
        # Convert tensors to numpy
        if isinstance(images[i], torch.Tensor):
            image = images[i].cpu().numpy()
            if image.shape[0] == 3:  # CHW format
                image = np.transpose(image, (1, 2, 0))
        else:
            image = images[i]
        
        if isinstance(targets[i], torch.Tensor):
            target = targets[i].cpu().numpy().squeeze()
        else:
            target = targets[i]
        
        if isinstance(predictions[i], torch.Tensor):
            prediction = torch.sigmoid(predictions[i]).cpu().numpy().squeeze()
        else:
            prediction = predictions[i]
        
        # Original image (use first channel if RGB)
        if len(image.shape) == 3:
            display_image = image[:, :, 0]
        else:
            display_image = image
        
        axes[i, 0].imshow(display_image, cmap='gray')
        axes[i, 0].set_title('SAR Image')
        axes[i, 0].axis('off')
        
        # Ground truth mask
        axes[i, 1].imshow(target, cmap='jet', alpha=0.7)
        axes[i, 1].set_title('Ground Truth')
        axes[i, 1].axis('off')
        
        # Predicted mask
        axes[i, 2].imshow(prediction, cmap='jet', alpha=0.7)
        axes[i, 2].set_title('Prediction')
        axes[i, 2].axis('off')
        
        # Overlay
        axes[i, 3].imshow(display_image, cmap='gray')
        axes[i, 3].imshow(prediction > 0.5, cmap='jet', alpha=0.5)
        axes[i, 3].set_title('Overlay')
        axes[i, 3].axis('off')
    
    plt.tight_layout()
    plt.show()

## 4. Segmentation Training Pipeline

In [ ]:
class SegmentationTrainer:
    """Training manager for segmentation model"""
    
    def __init__(self, model, train_loader, val_loader, config):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.config = config
        
        # Loss function
        self.criterion = get_segmentation_loss(config)
        
        # Optimizer
        optimizer_type = config.get('optimizer', 'adamw')
        if optimizer_type == 'adamw':
            self.optimizer = optim.AdamW(
                model.parameters(),
                lr=config.get('learning_rate', 1e-4),
                weight_decay=config.get('weight_decay', 1e-4)
            )
        elif optimizer_type == 'sgd':
            self.optimizer = optim.SGD(
                model.parameters(),
                lr=config.get('learning_rate', 1e-3),
                momentum=config.get('momentum', 0.9),
                weight_decay=config.get('weight_decay', 1e-4)
            )
        
        # Scheduler
        scheduler_type = config.get('scheduler', 'reduce_lr')
        if scheduler_type == 'reduce_lr':
            self.scheduler = ReduceLROnPlateau(
                self.optimizer, mode='min', factor=0.5, patience=5, verbose=True
            )
        elif scheduler_type == 'cosine':
            self.scheduler = CosineAnnealingLR(
                self.optimizer, T_max=config.get('epochs', 100)
            )
        
        # Early stopping
        self.early_stopping = EarlyStopping(
            patience=config.get('patience', 10),
            min_delta=config.get('min_delta', 1e-4)
        )
        
        # Training history
        self.history = {
            'train_loss': [],
            'val_loss': [],
            'train_iou': [],
            'val_iou': [],
            'train_dice': [],
            'val_dice': []
        }
    
    def train_epoch(self):
        """Train for one epoch"""
        self.model.train()
        running_loss = 0.0
        metrics_tracker = SegmentationMetrics()
        
        progress_bar = tqdm(self.train_loader, desc='Training')
        
        for batch_idx, (inputs, targets) in enumerate(progress_bar):
            inputs, targets = inputs.to(device), targets.to(device)
            
            self.optimizer.zero_grad()
            outputs = self.model(inputs)
            loss = self.criterion(outputs, targets)
            
            loss.backward()
            
            # Gradient clipping
            if self.config.get('grad_clip', 0) > 0:
                torch.nn.utils.clip_grad_norm_(
                    self.model.parameters(), self.config['grad_clip']
                )
            
            self.optimizer.step()
            
            # Statistics
            running_loss += loss.item()
            
            # Update metrics
            with torch.no_grad():
                predictions = torch.sigmoid(outputs)
                metrics_tracker.update(predictions, targets)
            
            # Update progress bar
            avg_metrics = metrics_tracker.get_average_metrics()
            progress_bar.set_postfix({
                'Loss': f'{running_loss/(batch_idx+1):.4f}',
                'IoU': f'{avg_metrics["iou"]:.4f}',
                'Dice': f'{avg_metrics["dice"]:.4f}'
            })
        
        epoch_loss = running_loss / len(self.train_loader)
        epoch_metrics = metrics_tracker.get_average_metrics()
        
        return epoch_loss, epoch_metrics
    
    def validate_epoch(self):
        """Validate for one epoch"""
        self.model.eval()
        running_loss = 0.0
        metrics_tracker = SegmentationMetrics()
        
        with torch.no_grad():
            progress_bar = tqdm(self.val_loader, desc='Validation')
            
            for batch_idx, (inputs, targets) in enumerate(progress_bar):
                inputs, targets = inputs.to(device), targets.to(device)
                
                outputs = self.model(inputs)
                loss = self.criterion(outputs, targets)
                
                running_loss += loss.item()
                
                # Update metrics
                predictions = torch.sigmoid(outputs)
                metrics_tracker.update(predictions, targets)
                
                # Update progress bar
                avg_metrics = metrics_tracker.get_average_metrics()
                progress_bar.set_postfix({
                    'Loss': f'{running_loss/(batch_idx+1):.4f}',
                    'IoU': f'{avg_metrics["iou"]:.4f}',
                    'Dice': f'{avg_metrics["dice"]:.4f}'
                })
        
        epoch_loss = running_loss / len(self.val_loader)
        epoch_metrics = metrics_tracker.get_average_metrics()
        
        return epoch_loss, epoch_metrics
    
    def train(self, epochs):
        """Full training loop"""
        print(f"Starting segmentation training for {epochs} epochs...")
        
        for epoch in range(epochs):
            print(f"\nEpoch {epoch+1}/{epochs}")
            print("-" * 60)
            
            # Train
            train_loss, train_metrics = self.train_epoch()
            
            # Validate
            val_loss, val_metrics = self.validate_epoch()
            
            # Update history
            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            self.history['train_iou'].append(train_metrics['iou'])
            self.history['val_iou'].append(val_metrics['iou'])
            self.history['train_dice'].append(train_metrics['dice'])
            self.history['val_dice'].append(val_metrics['dice'])
            
            # Scheduler step
            if isinstance(self.scheduler, ReduceLROnPlateau):
                self.scheduler.step(val_loss)
            else:
                self.scheduler.step()
            
            # Print results
            print(f"Train - Loss: {train_loss:.4f}, IoU: {train_metrics['iou']:.4f}, Dice: {train_metrics['dice']:.4f}")
            print(f"Val   - Loss: {val_loss:.4f}, IoU: {val_metrics['iou']:.4f}, Dice: {val_metrics['dice']:.4f}")
            
            # Early stopping
            if self.early_stopping(val_loss, self.model):
                print(f"Early stopping triggered after epoch {epoch+1}")
                break
        
        print("Segmentation training completed!")
        return self.history

# Early stopping class (reused from detection model)
class EarlyStopping:
    def __init__(self, patience=7, min_delta=0, restore_best_weights=True):
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best_weights = restore_best_weights
        self.best_loss = None
        self.counter = 0
        self.best_weights = None
    
    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.save_checkpoint(model)
        elif val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            self.save_checkpoint(model)
        else:
            self.counter += 1
        
        if self.counter >= self.patience:
            if self.restore_best_weights:
                model.load_state_dict(self.best_weights)
            return True
        return False
    
    def save_checkpoint(self, model):
        self.best_weights = model.state_dict().copy()

## 5. Model Configuration and Setup

In [ ]:
# Segmentation model configuration
SEGMENTATION_CONFIG = {
    # Model architecture
    'model_type': 'smp_unet',  # 'smp_unet', 'attention_unet'
    'architecture': 'unet',  # 'unet', 'unetplusplus', 'fpn', 'pspnet', 'deeplabv3plus'
    'encoder_name': 'resnet50',  # 'resnet50', 'efficientnet-b4', 'resnext50_32x4d'
    'encoder_weights': 'imagenet',
    'in_channels': 3,
    'classes': 1,  # Binary segmentation
    'activation': None,  # None for sigmoid in loss function
    'features': 64,  # For attention_unet
    
    # Training parameters
    'epochs': 150,
    'batch_size': 8,  # Smaller batch size for segmentation
    'learning_rate': 1e-4,
    'weight_decay': 1e-5,
    'optimizer': 'adamw',
    'scheduler': 'reduce_lr',
    
    # Loss function
    'loss_type': 'combined',  # 'dice', 'iou', 'focal', 'tversky', 'combined', 'bce'
    'dice_weight': 0.4,
    'focal_weight': 0.4,
    'iou_weight': 0.2,
    'focal_alpha': 1,
    'focal_gamma': 2,
    'tversky_alpha': 0.3,
    'tversky_beta': 0.7,
    'smooth': 1e-6,
    
    # Regularization
    'grad_clip': 1.0,
    
    # Early stopping
    'patience': 15,
    'min_delta': 1e-5,
    
    # Data
    'data_dir': '../data',
    'num_workers': 4,
    'threshold': 0.5  # For binary predictions
}

# Save configuration
with open('segmentation_config.json', 'w') as f:
    json.dump(SEGMENTATION_CONFIG, f, indent=2)

print("Segmentation model configuration:")
for key, value in SEGMENTATION_CONFIG.items():
    print(f"  {key}: {value}")

## 6. Model Creation and Testing

In [ ]:
# Create segmentation model
segmentation_model = create_segmentation_model(SEGMENTATION_CONFIG)
print(f"Created {SEGMENTATION_CONFIG['model_type']} model with {SEGMENTATION_CONFIG.get('encoder_name', 'custom')} encoder")

# Model summary
total_params = sum(p.numel() for p in segmentation_model.parameters())
trainable_params = sum(p.numel() for p in segmentation_model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Test forward pass
print("\nTesting forward pass...")
dummy_input = torch.randn(1, 3, 512, 512).to(device)
segmentation_model.eval()
with torch.no_grad():
    output = segmentation_model(dummy_input)
    print(f"Input shape: {dummy_input.shape}")
    print(f"Output shape: {output.shape}")
    
    # Apply sigmoid to get probabilities
    probs = torch.sigmoid(output)
    print(f"Output range: [{output.min():.4f}, {output.max():.4f}]")
    print(f"Probability range: [{probs.min():.4f}, {probs.max():.4f}]")

# Test loss functions
print("\nTesting loss functions...")
dummy_target = torch.randint(0, 2, (1, 512, 512)).float().to(device)

loss_fn = get_segmentation_loss(SEGMENTATION_CONFIG)
test_loss = loss_fn(output, dummy_target.unsqueeze(1))
print(f"Combined loss: {test_loss.item():.4f}")

# Test metrics
print("\nTesting metrics...")
test_metrics = calculate_metrics(probs, dummy_target.unsqueeze(1))
for metric, value in test_metrics.items():
    print(f"{metric}: {value:.4f}")

print("\nSegmentation model is ready for training!")
print("To train with real data:")
print("1. Replace dummy data with actual SAR dataset and masks")
print("2. Uncomment and run the training section below")

## 7. Training Loop (Uncomment when ready)

In [ ]:
# Uncomment this section when you have real data
"""
# Create data loaders with real data
data_loaders = create_data_loaders(
    SEGMENTATION_CONFIG['data_dir'], 
    batch_size=SEGMENTATION_CONFIG['batch_size'],
    num_workers=SEGMENTATION_CONFIG['num_workers']
)

train_loader = data_loaders['segmentation']['train']
val_loader = data_loaders['segmentation']['val']

# Create trainer
seg_trainer = SegmentationTrainer(segmentation_model, train_loader, val_loader, SEGMENTATION_CONFIG)

# Start training
seg_history = seg_trainer.train(SEGMENTATION_CONFIG['epochs'])

# Save model
torch.save({
    'model_state_dict': segmentation_model.state_dict(),
    'config': SEGMENTATION_CONFIG,
    'history': seg_history
}, 'oil_spill_segmentation.pth')

print("Segmentation model saved as 'oil_spill_segmentation.pth'")
"""

print("Training section is commented out - uncomment when ready with real data")
print("\nNext steps:")
print("1. Prepare segmentation masks for your SAR dataset")
print("2. Train the model with real data")
print("3. Evaluate results in the evaluation notebook")

## 8. Visualization Functions

In [ ]:
def plot_segmentation_history(history):
    """Plot training curves for segmentation"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Loss curves
    axes[0, 0].plot(history['train_loss'], label='Train Loss', marker='o')
    axes[0, 0].plot(history['val_loss'], label='Validation Loss', marker='s')
    axes[0, 0].set_title('Model Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # IoU curves
    axes[0, 1].plot(history['train_iou'], label='Train IoU', marker='o')
    axes[0, 1].plot(history['val_iou'], label='Validation IoU', marker='s')
    axes[0, 1].set_title('IoU Score')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('IoU')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Dice curves
    axes[1, 0].plot(history['train_dice'], label='Train Dice', marker='o')
    axes[1, 0].plot(history['val_dice'], label='Validation Dice', marker='s')
    axes[1, 0].set_title('Dice Score')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Dice')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Combined metrics
    axes[1, 1].plot(history['val_iou'], label='Val IoU', marker='s')
    axes[1, 1].plot(history['val_dice'], label='Val Dice', marker='^')
    axes[1, 1].set_title('Validation Metrics')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Score')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def create_dummy_segmentation_data():
    """Create dummy data for visualization testing"""
    # Create dummy SAR images
    images = []
    targets = []
    predictions = []
    
    for i in range(4):
        # Dummy SAR image
        image = np.random.randn(3, 512, 512) * 0.5 + 0.5
        image = np.clip(image, 0, 1)
        
        # Dummy oil spill mask (circular region)
        mask = np.zeros((512, 512))
        center_x, center_y = np.random.randint(100, 412, 2)
        radius = np.random.randint(30, 80)
        
        y, x = np.ogrid[:512, :512]
        mask_region = (x - center_x) ** 2 + (y - center_y) ** 2 <= radius ** 2
        mask[mask_region] = 1
        
        # Add some noise to prediction
        prediction = mask + np.random.randn(512, 512) * 0.1
        prediction = np.clip(prediction, 0, 1)
        
        images.append(torch.tensor(image))
        targets.append(torch.tensor(mask))
        predictions.append(torch.tensor(prediction))
    
    return images, targets, predictions

# Test visualization with dummy data
print("Testing segmentation visualization...")
dummy_images, dummy_targets, dummy_predictions = create_dummy_segmentation_data()
visualize_segmentation_results(dummy_images, dummy_targets, dummy_predictions, num_samples=4)

print("\nSegmentation model notebook complete!")
print("Ready for the next step: Evaluation and Visualization notebook")